# 강의 05 · 실습 3 — 운영 장치 · (4) 고난도 I


## 1. 문제상황

- 구름월드 운영팀은 분기마다 「1차 모델 장애 훈련」을 합니다. 훈련에서는 1차 모델 이름을 일부러 존재하지 않는 이름으로 바꾸고, 그날 들어온 질문 5개를 대체(fallback)를 붙여 처리합니다.
- 지난 훈련에서는 대체가 발동하는 것만 확인하고 끝냈습니다. 어느 모델이 답했는지, 장애 때문에 비용이 얼마나 더 들었는지는 아무도 계산하지 않았습니다.
- 경영진은 「장애 하루에 안내 비용이 몇 배가 되는가」를 숫자로 묻습니다.
- 운영팀은 질문마다 답한 모델·비용·걸린 시간을 표로 만들고, 장애가 없을 때의 비용과 견주어 훈련 보고서를 내야 합니다.


## 2. 문제와 목표

- **문제**: 장애 훈련이 「대체가 발동했다」에서 끝나고, 장애가 비용과 시간에 미친 영향을 숫자로 남기지 않습니다.
- **목표**
  - 같은 질문 5개를 정상 상태와 장애 상태로 각각 처리합니다.
    - 두 상태: 정상은 1차 기본 모델이고 대체가 없으며, 장애는 1차에 존재하지 않는 이름을 넣고 대체 순서가 경량 → 고성능입니다
    - 질문 5개는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다
    - 존재하지 않는 모델 이름 `GHOST`(장애 상태의 1차 호출을 일부러 실패시키는 용도)도 단계 0에 주어져 있습니다
  - 질문마다 답한 모델·비용·걸린 시간을 표로 출력합니다.
  - 두 상태의 총비용과 배율, 장애 상태에서 답한 모델의 분포를 훈련 보고서 형식으로 출력합니다.
    - 배율: 장애 상태 총비용 ÷ 정상 상태 총비용
- **목표 달성 여부의 판정 기준**:
  - 질문 5개 × 상태 2개 = 10줄의 표가 출력되고,
  - 정상 상태는 전부 `gpt-5.6-luna`가 답하고 장애 상태는 전부 `gpt-4o-mini`로 시작하는 이름의 모델이 답하며,
  - 보고서 마지막에 장애 상태 총비용을 정상 상태 총비용으로 나눈 배율이 출력됩니다.
  - 장애 상태의 실패 시도가 LangSmith에 실패 런으로 남습니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex03_s4_diagram.svg)


## 4. 단계별 요구사항

1. **대체(fallback)를 붙인 호출 함수를 만듭니다.**
    - `call_and_measure(model, question, fallbacks)`는 모델을 한 번 부르고 답한 모델(`res.model`)·비용(`completion_cost`)·걸린 시간(초)·답을 딕셔너리로 돌려줍니다.
    - `fallbacks`가 있으면 호출 인자로 넘깁니다.
2. **비용과 시간을 잽니다.**
    - 질문 5개를 정상 상태(`PRIMARY`, 대체 없음)와 장애 상태(`GHOST`, 대체 `[CHEAP, HIGH]`)로 각각 처리해, 상태·질문·답한 모델·비용·초를 한 줄씩 출력합니다.
    - 질문 5개는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - 표를 다 출력한 뒤 `Client().flush()`를 불러 남은 런을 LangSmith로 보냅니다.
3. **기준값과 배율을 계산합니다.**
    - 정상 상태 총비용을 기준값으로 두고, 장애 상태 총비용을 기준값으로 나눈 배율을 계산합니다.
4. **훈련 보고서를 출력합니다.**
    - 상태별 총비용, 배율(장애 ÷ 정상, 소수 둘째 자리), 장애 상태에서 답한 모델의 분포(모델 이름: 질문 수)를 출력합니다.
    - 배율은 「장애 상태 총비용 / 정상 상태 총비용 = N배」 형식으로 출력합니다.


## 5. 코드 골격

골격은 네 단계입니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 대체(fallback) | 호출 함수에 `fallbacks` 인자를 붙입니다 | `call_and_measure(..., fallbacks=[CHEAP, HIGH])` | 1 |
| ② 비용·시간 측정 | 질문 5개 × 상태 2개의 비용과 시간을 잽니다 | `completion_cost`, `time.perf_counter()`, `Client().flush()` | 2 |
| ③ 기준값·배율 | 정상 상태 총비용을 기준값으로 두고 배율을 계산합니다 | 총비용 합산 | 3 |
| ④ 훈련 보고서 | 상태별 총비용·배율·답한 모델 분포를 출력합니다 | 모델별 질문 수 | 4 |


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델 이름 세 개를 정합니다. 이 실습의 모델 호출은 `litellm.completion`을 직접 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- 모델 세 개는 기본 모델(`PRIMARY`), 경량 모델(`CHEAP`), 고성능 모델(`HIGH`)입니다. 모델 이름은 공급자 이름을 앞에 붙인 문자열 그대로 쓰고, 별칭이나 중계 서버는 쓰지 않습니다.
- `litellm.suppress_debug_info = True`와 `logging` 설정 한 줄은 오류가 났을 때 litellm이 화면에 출력하는 안내 배너와 오류 로그를 끕니다. 동작에는 영향이 없습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
import logging
import os
import time
import warnings

from dotenv import load_dotenv, find_dotenv

import litellm
from langsmith import Client, traceable
from langsmith.run_helpers import get_current_run_tree

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")   # 추적 라이브러리가 내는 직렬화 경고를 화면에서 감춥니다
litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.CRITICAL)

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex03"

PRIMARY = "openai/gpt-5.6-luna"
CHEAP = "openai/gpt-4o-mini"
HIGH = "openai/gpt-5.6-terra"
print("모델 세 개:", PRIMARY, CHEAP, HIGH)

GHOST = "openai/gpt-5.6-luna-nonexistent"   # 존재하지 않는 모델 이름 — 장애 상태의 1차 호출을 일부러 실패시키는 용도
# 주어진 자료: 처리할 질문 목록 QUESTIONS — 값을 그대로 씁니다
QUESTIONS = ["자유이용권 환불이 되나요?", "운영 시간이 어떻게 되나요?", "야간개장 때 퍼레이드 하나요?",
             "주차 요금은 얼마인가요?", "안녕하세요!"]


운영 장치를 붙일 안내 서비스입니다. 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. `make_messages`가 질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만들고, 모델 호출 함수 `litellm.completion`은 `traceable`로 감싸 llm 런에 기록되어 있습니다(답한 모델 이름을 런 메타데이터에 적습니다). 서비스가 도는 것을 먼저 확인합니다.


In [ ]:
FAQ = """
[환불] Q: 자유이용권 환불 규정 알려 주세요
A: 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.
[운영] Q: 운영 시간이 어떻게 되나요?
A: 평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다.
[야간] Q: 야간개장은 언제 하나요?
A: 금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.
[주차] Q: 주차 요금은 얼마인가요?
A: 자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.
"""

GUIDE = ("너는 놀이공원 구름월드의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
         "인사말에는 짧은 인사로 답한다. FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다.")


def make_messages(question: str, guide: str = GUIDE) -> list:
    """질문을 받아 FAQ를 근거로 넣은 메시지 목록을 만든다."""
    return [{"role": "system", "content": guide + "\n=== FAQ ===\n" + FAQ},
            {"role": "user", "content": question}]


_completion = litellm.completion


@traceable(run_type="llm", name="litellm.completion", metadata={"ls_provider": "openai"})
def completion(**kwargs):
    """이 실습에서 쓰는 관측 계측. 답한 모델 이름을 런 메타데이터에 적는다."""
    res = _completion(**kwargs)
    get_current_run_tree().metadata["ls_model_name"] = res.model
    return res


litellm.completion = completion

Q = "자유이용권 환불이 되나요?"
res = litellm.completion(model=PRIMARY, messages=make_messages(Q))
print(res.model, "→", res.choices[0].message.content[:60])

### 단계 ① — 대체(fallback) (요구사항 1)

호출 함수 하나가 대체 인자와 측정을 함께 맡습니다.


In [ ]:
# 여기에 단계 ①을 작성합니다.

### 단계 ② — 비용·시간 측정 (요구사항 2)

질문 5개를 두 상태로 처리해 표에 모읍니다.


In [ ]:
# 여기에 단계 ②을 작성합니다.

### 단계 ③ — 기준값·배율 (요구사항 3)

정상 상태 총비용이 기준값입니다.


In [ ]:
# 여기에 단계 ③을 작성합니다.

### 단계 ④ — 훈련 보고서 (요구사항 4)

보고서를 출력합니다. 답한 모델 분포가 곧 장애 때의 티어링 결과입니다.


In [ ]:
# 여기에 단계 ④을 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 단계 ② 표에 정상 5줄·장애 5줄이 출력되고, 정상은 전부 `gpt-5.6-luna`, 장애는 전부 `gpt-4o-mini`로 시작하는 이름의 모델이 답했습니다.
2. 걸린 시간 열에는 두 효과가 섞여 있습니다. 장애 상태는 1차 실패에 든 시간이 더해지지만, 2차 경량 모델이 기본 모델보다 빨라 정상보다 짧게 나올 수도 있습니다. 두 효과를 함께 읽습니다.
3. 단계 ④ 보고서에 두 상태의 총비용과 배율이 출력됩니다. 대체 순서에서 경량 모델이 먼저 받으므로 배율이 1보다 작습니다.
4. LangSmith 프로젝트 `sesac-lec05-ex03`에 장애 상태의 실패 런 5개가 남습니다.

네 가지가 모두 확인되면 완성입니다. 배율이 1보다 작다는 결과를 보고 「장애가 비용을 줄였는가」를 다음 질문으로 이어 갑니다. 답의 품질은 같지 않을 수 있습니다.
